# Caso de uso de clusterização com KMeans
## A Batalha dos vizinhos em Toronto

Pacotes necessários para execução do caso de uso (instalação via linha de comando):
+ conda install -c conda-forge **geopy**
+ conda install -c anaconda **requests**
+ conda install -c conda-forge **folium**
+ conda install -c conda-forge **lxml**
+ conda install -c anaconda **html5lib**
+ conda install -c anaconda **beautifulsoup4**

<b>Objetivos - Parte 1</b>:

1) O dataframe de dados possui em três colunas: PostalCode, Borough e Neighbourhood


2) Processar apenas os exemplos que têm um bairro (Borough) atribuído. Ignorar os exemplos que não tem um bairro atribuído.

3) Pode existir mais de uma vizinhança no código postal de uma área. Por exemplo, na tabela da página da Wikipedia, você vai notar que M5A tem duas vizinhanças: Regent Park e Harbourfront. Essas duas linhas são combinadas em uma linha com os nomes separados por uma vírgula.

4) Se uma célula tiver um bairro (Borough), mas uma vizinhança (Neighborhood) não atribuída (Not assigned), a vizinhança terá o mesmo valor que o bairro.

In [15]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
df = pd.read_html('https://en.wikipedia.org/w/index.php?title=List_of_postal_codes_of_Canada:_M&oldid=977312452')[0]

df.columns = ['Postal Code','Borough','Neighborhood']
df

,Postal Code,Borough,Neighborhood
0,M1A,Not assigned,Not assigned
1,M2A,Not assigned,Not assigned
2,M3A,North York,Parkwoods
3,M4A,North York,Victoria Village
4,M5A,Downtown Toronto,"Regent Park, Harbourfront"
...,...,...,...
175,M5Z,Not assigned,Not assigned
176,M6Z,Not assigned,Not assigned
177,M7Z,Not assigned,Not assigned
178,M8Z,Etobicoke,"Mimico NW, The Queensway West, South of Bloor,..."


In [16]:
df = df[df.Borough != 'Not assigned']
df


,Postal Code,Borough,Neighborhood
2,M3A,North York,Parkwoods
3,M4A,North York,Victoria Village
4,M5A,Downtown Toronto,"Regent Park, Harbourfront"
5,M6A,North York,"Lawrence Manor, Lawrence Heights"
6,M7A,Downtown Toronto,"Queen's Park, Ontario Provincial Government"
...,...,...,...
160,M8X,Etobicoke,"The Kingsway, Montgomery Road, Old Mill North"
165,M4Y,Downtown Toronto,Church and Wellesley
168,M7Y,East Toronto,"Business reply mail Processing Centre, South C..."
169,M8Y,Etobicoke,"Old Mill South, King's Mill Park, Sunnylea, Hu..."


In [17]:
df.loc[df['Neighborhood'] =='Not assigned' , 'Neighborhood'] = df['Borough']
df

,Postal Code,Borough,Neighborhood
2,M3A,North York,Parkwoods
3,M4A,North York,Victoria Village
4,M5A,Downtown Toronto,"Regent Park, Harbourfront"
5,M6A,North York,"Lawrence Manor, Lawrence Heights"
6,M7A,Downtown Toronto,"Queen's Park, Ontario Provincial Government"
...,...,...,...
160,M8X,Etobicoke,"The Kingsway, Montgomery Road, Old Mill North"
165,M4Y,Downtown Toronto,Church and Wellesley
168,M7Y,East Toronto,"Business reply mail Processing Centre, South C..."
169,M8Y,Etobicoke,"Old Mill South, King's Mill Park, Sunnylea, Hu..."


In [18]:
# Resetar os índices
df.groupby(['Postal Code','Borough'])['Neighborhood'].apply(lambda x: ','.join(x.astype(str))).reset_index()

,Postal Code,Borough,Neighborhood
0,M1B,Scarborough,"Malvern, Rouge"
1,M1C,Scarborough,"Rouge Hill, Port Union, Highland Creek"
2,M1E,Scarborough,"Guildwood, Morningside, West Hill"
3,M1G,Scarborough,Woburn
4,M1H,Scarborough,Cedarbrae
...,...,...,...
98,M9N,York,Weston
99,M9P,Etobicoke,Westmount
100,M9R,Etobicoke,"Kingsview Village, St. Phillips, Martin Grove ..."
101,M9V,Etobicoke,"South Steeles, Silverstone, Humbergate, Jamest..."


In [19]:
df.shape

(103, 3)

<b>Objetivos - Parte 2</b>:

1) Agora que temos um dataframe do código postal de cada vizinhança, junto com o nome do bairro, podemos utilizar os dados de localização do Foursquare.

2) Explorar e clusterizar os bairros de Toronto.


<b> Importar as latitudes e longitudes, de acordo com o código postal </b>

In [23]:
df1 = pd.read_csv("https://cocl.us/Geospatial_data")
df1

,Postal Code,Latitude,Longitude
0,M1B,43.806686,-79.194353
1,M1C,43.784535,-79.160497
2,M1E,43.763573,-79.188711
3,M1G,43.770992,-79.216917
4,M1H,43.773136,-79.239476
...,...,...,...
98,M9N,43.706876,-79.518188
99,M9P,43.696319,-79.532242
100,M9R,43.688905,-79.554724
101,M9V,43.739416,-79.588437


In [24]:
df = pd.merge(df, df1, on = "Postal Code")
df

,Postal Code,Borough,Neighborhood,Latitude,Longitude
0,M3A,North York,Parkwoods,43.753259,-79.329656
1,M4A,North York,Victoria Village,43.725882,-79.315572
2,M5A,Downtown Toronto,"Regent Park, Harbourfront",43.654260,-79.360636
3,M6A,North York,"Lawrence Manor, Lawrence Heights",43.718518,-79.464763
4,M7A,Downtown Toronto,"Queen's Park, Ontario Provincial Government",43.662301,-79.389494
...,...,...,...,...,...
98,M8X,Etobicoke,"The Kingsway, Montgomery Road, Old Mill North",43.653654,-79.506944
99,M4Y,Downtown Toronto,Church and Wellesley,43.665860,-79.383160
100,M7Y,East Toronto,"Business reply mail Processing Centre, South C...",43.662744,-79.321558
101,M8Y,Etobicoke,"Old Mill South, King's Mill Park, Sunnylea, Hu...",43.636258,-79.498509


<b>Objetivos - Parte 3</b>:

Gerar mapa para visualizar os bairros e como eles se agrupam.

<b> Importar os pacotes necessários para visualizar e analisar o mapa </b>

In [25]:

from geopy.geocoders import Nominatim 

import requests 
from pandas.io.json import json_normalize 

import matplotlib.cm as cm
import matplotlib.colors as colors

from sklearn.cluster import KMeans

import folium 

print('Pacotes importados.')

Pacotes importados.


In [26]:
#Obter as coordenadas geográficas de Toronto, no Canadá.

address = 'Toronto, ON, Canada'

geolocator = Nominatim(user_agent="to_explorer")
location = geolocator.geocode(address)
latitude = location.latitude
longitude = location.longitude
print('As coordenadas geográficas de Toronto, Canadá são {}, {}.'.format(latitude, longitude))

As coordenadas geográficas de Toronto, Canadá são 43.6534817, -79.3839347.


In [28]:
# Criar um mapa de Toronto usando os valores de latitude e longitude.
map_Toronto = folium.Map(location=[latitude,longitude], zoom_start=10)

# Adicionar marcadores ao mapa
for lat, lng, borough, neighborhood in zip(df['Latitude'], df['Longitude'], df['Borough'], df['Neighborhood']):
    label = '{}, {}'.format(neighborhood,borough)
    label = folium.Popup(label, parse_html=True)
    folium.CircleMarker(
        [lat, lng],
        radius=5,
        popup=label,
        color='blue',
        fill=True,
        fill_color='#3186cc',
        fill_opacity=0.7,
        parse_html=False).add_to(map_Toronto)  
    
map_Toronto

![map1.JPG](https://raw.githubusercontent.com/sarthak815/Coursera_Capstone/master/map1.JPG)

<b>Definir as credenciais de acesso ao Foursquare:</b>

Crie as suas próprias credenciais no link: https://developer.foursquare.com

In [29]:
# Credenciais do Foursquare
CLIENT_ID = 'EILREA14SNMZ0CWXNM0ND3WL5RIVV1B41U1NYADE24SPTYOX'
CLIENT_SECRET = 'FNAPNA3MSBWLU2YZ15RH4VQPI0OS5MAFS0ARQTDK1VDZM4JR'
VERSION = '20180604'

In [30]:
# definir raio e limite de locais para buscar
radius=500
LIMIT=100

In [31]:

def getNearbyVenues(names, latitudes, longitudes, radius=500):
    
    venues_list=[]
    for name, lat, lng in zip(names, latitudes, longitudes):
        print(name)
            
        # criar a URL para acessar a API
        url = 'https://api.foursquare.com/v2/venues/explore?&client_id={}&client_secret={}&v={}&ll={},{}&radius={}&limit={}'.format(
            CLIENT_ID, 
            CLIENT_SECRET, 
            VERSION, 
            lat, 
            lng, 
            radius, 
            LIMIT)
            
        # fazer a requisição GET
        results = requests.get(url).json()["response"]['groups'][0]['items']
        
        # retornar apenas as informações relevantes para cada local próximo
        venues_list.append([(
            name, 
            lat, 
            lng, 
            v['venue']['name'], 
            v['venue']['location']['lat'], 
            v['venue']['location']['lng'],  
            v['venue']['categories'][0]['name']) for v in results])

    nearby_venues = pd.DataFrame([item for venue_list in venues_list for item in venue_list])
    nearby_venues.columns = ['Neighbourhood', 
                  'Neighbourhood Latitude', 
                  'Neighbourhood Longitude', 
                  'Venue', 
                  'Venue Latitude', 
                  'Venue Longitude', 
                  'Venue Category']
    
    return(nearby_venues)

In [32]:

toronto_venues = getNearbyVenues(names=df['Neighborhood'],
                                   latitudes=df['Latitude'],
                                   longitudes=df['Longitude']
                                  )

Parkwoods
Victoria Village
Regent Park, Harbourfront
Lawrence Manor, Lawrence Heights
Queen's Park, Ontario Provincial Government
Islington Avenue, Humber Valley Village
Malvern, Rouge
Don Mills
Parkview Hill, Woodbine Gardens
Garden District, Ryerson
Glencairn
West Deane Park, Princess Gardens, Martin Grove, Islington, Cloverdale
Rouge Hill, Port Union, Highland Creek
Don Mills
Woodbine Heights
St. James Town
Humewood-Cedarvale
Eringate, Bloordale Gardens, Old Burnhamthorpe, Markland Wood
Guildwood, Morningside, West Hill
The Beaches
Berczy Park
Caledonia-Fairbanks
Woburn
Leaside
Central Bay Street
Christie
Cedarbrae
Hillcrest Village
Bathurst Manor, Wilson Heights, Downsview North
Thorncliffe Park
Richmond, Adelaide, King
Dufferin, Dovercourt Village
Scarborough Village
Fairview, Henry Farm, Oriole
Northwood Park, York University
East Toronto, Broadview North (Old East York)
Harbourfront East, Union Station, Toronto Islands
Little Portugal, Trinity
Kennedy Park, Ionview, East Birchmo

<b>Checando o tamanho do dataframe:</b>

In [33]:
print(toronto_venues.shape)
toronto_venues.head()

(2313, 7)


,Neighbourhood,Neighbourhood Latitude,Neighbourhood Longitude,Venue,Venue Latitude,Venue Longitude,Venue Category
0,Parkwoods,43.753259,-79.329656,Brookbanks Park,43.754751,-79.328439,Park
1,Parkwoods,43.753259,-79.329656,Variety Store,43.752018,-79.332833,Food and Beverage Retail
2,Victoria Village,43.725882,-79.315572,Victoria Village Arena,43.723321,-79.315512,Hockey Stadium
3,Victoria Village,43.725882,-79.315572,Tim Hortons,43.725517,-79.313103,Coffee Shop
4,Victoria Village,43.725882,-79.315572,Portugril,43.725482,-79.312809,Portuguese Restaurant


<b>O número de locais por vizinhança:</b>

In [34]:
toronto_venues.groupby('Neighbourhood').count()

,Neighbourhood Latitude,Neighbourhood Longitude,Venue,Venue Latitude,Venue Longitude,Venue Category
Neighbourhood,,,,,,
Agincourt,2,2,2,2,2,2
"Alderwood, Long Branch",7,7,7,7,7,7
"Bathurst Manor, Wilson Heights, Downsview North",17,17,17,17,17,17
Bayview Village,3,3,3,3,3,3
"Bedford Park, Lawrence Manor East",20,20,20,20,20,20
...,...,...,...,...,...,...
"Willowdale, Willowdale East",35,35,35,35,35,35
"Willowdale, Willowdale West",5,5,5,5,5,5
Woburn,3,3,3,3,3,3


<b>Vamos ver quantas categorias exclusivas podem ser selecionadas de todos os locais</b>

In [35]:
print('Existem {} categorias únicas.'.format(len(toronto_venues['Venue Category'].unique())))

Existem 273 categorias únicas.


<b> Analisando cada vizinhança</b>

In [36]:
# codificação one hot
toronto_onehot = pd.get_dummies(toronto_venues[['Venue Category']], prefix="", prefix_sep="")

# adicionar a coluna vizinhança novamente ao dataframe
toronto_onehot['Neighborhood'] = toronto_venues['Neighbourhood'] 

fixed_columns = [toronto_onehot.columns[-1]] + list(toronto_onehot.columns[:-1])
toronto_onehot = toronto_onehot[fixed_columns]

toronto_onehot.head()

,Women's Store,Adult Store,Afghan Restaurant,Airport,Airport Food Court,Airport Gate,Airport Lounge,Airport Service,Airport Terminal,American Restaurant,...,Travel and Transportation,Vegan and Vegetarian Restaurant,Video Games Store,Vietnamese Restaurant,Vintage and Thrift Store,Warehouse or Wholesale Store,Whisky Bar,Wine Bar,Wine Store,Wings Joint
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:

toronto_onehot.shape

(2313, 273)

<b>A seguir, vamos agrupar as linhas por vizinhança e calcular a média da frequência de cada categoria</b>


In [38]:
toronto_grouped = toronto_onehot.groupby('Neighborhood').mean().reset_index()
toronto_grouped

,Neighborhood,Women's Store,Adult Store,Afghan Restaurant,Airport,Airport Food Court,Airport Gate,Airport Lounge,Airport Service,Airport Terminal,...,Travel and Transportation,Vegan and Vegetarian Restaurant,Video Games Store,Vietnamese Restaurant,Vintage and Thrift Store,Warehouse or Wholesale Store,Whisky Bar,Wine Bar,Wine Store,Wings Joint
0,Agincourt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,"Alderwood, Long Branch",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,"Bathurst Manor, Wilson Heights, Downsview North",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,Bayview Village,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,"Bedford Park, Lawrence Manor East",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.05,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,"Willowdale, Willowdale East",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.057143,0.0,0.0,0.0,0.0,0.0,0.0
92,"Willowdale, Willowdale West",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
93,Woburn,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
94,Woodbine Heights,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0



<b>Vamos imprimir cada vizinhança junto com os 5 locais mais comuns</b>

In [39]:
num_top_venues = 5

for hood in toronto_grouped['Neighborhood']:
    print("----"+hood+"----")
    temp = toronto_grouped[toronto_grouped['Neighborhood'] == hood].T.reset_index()
    temp.columns = ['venue','freq']
    temp = temp.iloc[1:]
    temp['freq'] = temp['freq'].astype(float)
    temp = temp.round({'freq': 2})
    print(temp.sort_values('freq', ascending=False).reset_index(drop=True).head(num_top_venues))
    print('\n')

----Agincourt----
                 venue  freq
0       Clothing Store   0.5
1       Breakfast Spot   0.5
2        Women's Store   0.0
3                Motel   0.0
4  Moroccan Restaurant   0.0


----Alderwood, Long Branch----
           venue  freq
0       Pizzeria  0.29
1    Coffee Shop  0.14
2  Sandwich Spot  0.14
3     Playground  0.14
4   Burger Joint  0.14


----Bathurst Manor, Wilson Heights, Downsview North----
                       venue  freq
0                Coffee Shop  0.12
1                       Deli  0.06
2              Shopping Mall  0.06
3  Middle Eastern Restaurant  0.06
4              Sandwich Spot  0.06


----Bayview Village----
                venue  freq
0  Chinese Restaurant  0.33
1                Café  0.33
2      Shopping Plaza  0.33
3       Women's Store  0.00
4  Mobile Phone Store  0.00


----Bedford Park, Lawrence Manor East----
                venue  freq
0        Fuel Station  0.10
1     Thai Restaurant  0.10
2  Persian Restaurant  0.05
3    Greek Restaura

              venue  freq
0       Coffee Shop  0.07
1              Café  0.07
2              Park  0.04
3  Ice Cream Parlor  0.04
4          Pizzeria  0.04


----Steeles West, L'Amoreaux West----
                  venue  freq
0  Fast Food Restaurant  0.13
1              Pharmacy  0.07
2           Coffee Shop  0.07
3       Bubble Tea Shop  0.07
4          Burger Joint  0.07


----Stn A PO Boxes----
                venue  freq
0         Coffee Shop  0.10
1                 Pub  0.04
2              Bakery  0.04
3               Hotel  0.03
4  Seafood Restaurant  0.03


----Studio District----
                venue  freq
0         Coffee Shop  0.08
1                Café  0.08
2  Seafood Restaurant  0.05
3              Bakery  0.05
4         Music Store  0.05


----Summerhill West, Rathnelly, South Hill, Forest Hill SE, Deer Park----
                 venue  freq
0           Bagel Shop  0.15
1                  Pub  0.15
2         Liquor Store  0.08
3  American Restaurant  0.08
4           Rest

<b>Vamos colocar essa informação em um dataframe</b>

In [40]:
def return_most_common_venues(row, num_top_venues):
    row_categories = row.iloc[1:]
    row_categories_sorted = row_categories.sort_values(ascending=False)
    
    return row_categories_sorted.index.values[0:num_top_venues]

In [41]:
import numpy as np

num_top_venues = 10

indicators = ['st', 'nd', 'rd']

# create columns according to number of top venues
columns = ['Neighborhood']
for ind in np.arange(num_top_venues):
    try:
        columns.append('{}{} Most Common Venue'.format(ind+1, indicators[ind]))
    except:
        columns.append('{}th Most Common Venue'.format(ind+1))

# create a new dataframe
neighborhoods_venues_sorted = pd.DataFrame(columns=columns)
neighborhoods_venues_sorted['Neighborhood'] = toronto_grouped['Neighborhood']

for ind in np.arange(toronto_grouped.shape[0]):
    neighborhoods_venues_sorted.iloc[ind, 1:] = return_most_common_venues(toronto_grouped.iloc[ind, :], num_top_venues)

neighborhoods_venues_sorted.head()

,Neighborhood,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
0,Agincourt,Clothing Store,Breakfast Spot,Escape Room,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store,Empanada Restaurant,Wings Joint
1,"Alderwood, Long Branch",Pizzeria,Burger Joint,Coffee Shop,Pub,Playground,Sandwich Spot,Eastern European Restaurant,Doner Restaurant,Donut Shop,Drugstore
2,"Bathurst Manor, Wilson Heights, Downsview North",Coffee Shop,Chinese Restaurant,Shopping Mall,Bridal Store,Sandwich Spot,Supermarket,Restaurant,Sushi Restaurant,Middle Eastern Restaurant,Ice Cream Parlor
3,Bayview Village,Café,Shopping Plaza,Chinese Restaurant,Electronics Store,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Wings Joint
4,"Bedford Park, Lawrence Manor East",Thai Restaurant,Fuel Station,Coffee Shop,Indian Restaurant,Italian Restaurant,Liquor Store,Comfort Food Restaurant,Grocery Store,Persian Restaurant,Pharmacy


<b> Realizar a clusterização da vizinhança</b>

In [42]:
# Número de clusters
kclusters = 5


toronto_grouped_clustering = toronto_grouped.drop('Neighborhood', 1)

kmeans = KMeans(n_clusters=kclusters, random_state=0).fit(toronto_grouped_clustering)

# checar os rótulos do cluster gerado para cada linha do dataframe
#kmeans.labels_[0:10] 
kmeans.labels_

array([3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 4, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 1])

In [43]:
# Adicionar os rótulos do cluster
neighborhoods_venues_sorted.insert(0, 'Cluster_Labels', kmeans.labels_)

toronto_merged = df

# Juntar o datagrame toronto_grouped com toronto_data para adicionar as informações de 
# latitude e longitude para cada vizinhança
toronto_merged = toronto_merged.join(neighborhoods_venues_sorted.set_index('Neighborhood'), on='Neighborhood')

toronto_merged.head()

,Postal Code,Borough,Neighborhood,Latitude,Longitude,Cluster_Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
0,M3A,North York,Parkwoods,43.753259,-79.329656,1.0,Park,Food and Beverage Retail,Wings Joint,Electronics Store,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Empanada Restaurant
1,M4A,North York,Victoria Village,43.725882,-79.315572,0.0,Fuel Station,Coffee Shop,Hockey Stadium,Portuguese Restaurant,Electronics Store,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant
2,M5A,Downtown Toronto,"Regent Park, Harbourfront",43.654260,-79.360636,0.0,Coffee Shop,Pub,Italian Restaurant,Park,Performing Arts Venue,Bakery,Art Gallery,Thai Restaurant,Theater,Beer Store
3,M6A,North York,"Lawrence Manor, Lawrence Heights",43.718518,-79.464763,0.0,Furniture and Home Store,Clothing Store,Fashion Accessories Store,Party Supply Store,Boutique,Event Space,Vietnamese Restaurant,Coffee Shop,Flea Market,Education
4,M7A,Downtown Toronto,"Queen's Park, Ontario Provincial Government",43.662301,-79.389494,0.0,Coffee Shop,Sushi Restaurant,Fried Chicken Joint,Burger Joint,Café,Wings Joint,Seafood Restaurant,Sandwich Spot,Bubble Tea Shop,Restaurant


In [44]:
toronto_merged=toronto_merged.dropna()

In [45]:

toronto_merged['Cluster_Labels'] = toronto_merged.Cluster_Labels.astype(int)

**Visualizar os clusters**

In [46]:
# Criar o mapa
map_clusters = folium.Map(location=[latitude, longitude], zoom_start=11)

# Alterar as cores dos clusters
x = np.arange(kclusters)
ys = [i + x + (i*x)**2 for i in range(kclusters)]
colors_array = cm.rainbow(np.linspace(0, 1, len(ys)))
rainbow = [colors.rgb2hex(i) for i in colors_array]

# Adicionar os marcadores no mapa
markers_colors = []
for lat, lon, poi, cluster in zip(toronto_merged['Latitude'], toronto_merged['Longitude'], toronto_merged['Neighborhood'], toronto_merged['Cluster_Labels']):
    label = folium.Popup(str(poi) + ' Cluster ' + str(cluster), parse_html=True)
    folium.CircleMarker(
        [lat, lon],
        radius=5,
        popup=label,
        color=rainbow[cluster-1],
        fill=True,
        fill_color=rainbow[cluster-1],
        fill_opacity=0.7).add_to(map_clusters)
       
map_clusters

![map2.JPG](https://raw.githubusercontent.com/sarthak815/Coursera_Capstone/master/map2.JPG)


<b>Examinar os Clusters</b>

Cluster 0

In [47]:
toronto_merged.loc[toronto_merged['Cluster_Labels'] == 0, toronto_merged.columns[[1] + list(range(5, toronto_merged.shape[1]))]]

,Borough,Cluster_Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
1,North York,0,Fuel Station,Coffee Shop,Hockey Stadium,Portuguese Restaurant,Electronics Store,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant
2,Downtown Toronto,0,Coffee Shop,Pub,Italian Restaurant,Park,Performing Arts Venue,Bakery,Art Gallery,Thai Restaurant,Theater,Beer Store
3,North York,0,Furniture and Home Store,Clothing Store,Fashion Accessories Store,Party Supply Store,Boutique,Event Space,Vietnamese Restaurant,Coffee Shop,Flea Market,Education
4,Downtown Toronto,0,Coffee Shop,Sushi Restaurant,Fried Chicken Joint,Burger Joint,Café,Wings Joint,Seafood Restaurant,Sandwich Spot,Bubble Tea Shop,Restaurant
6,Scarborough,0,Fast Food Restaurant,Fuel Station,Falafel Restaurant,Eyecare Store,Exhibit,Event Space,Ethiopian Restaurant,Escape Room,Fashion Accessories Store,Electronics Store
...,...,...,...,...,...,...,...,...,...,...,...,...
97,Downtown Toronto,0,Coffee Shop,Café,Hotel,Restaurant,Deli,American Restaurant,Cocktail Bar,Thai Restaurant,Seafood Restaurant,Speakeasy
98,Etobicoke,0,River,Electronics Store,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Empanada Restaurant,Concert Hall
99,Downtown Toronto,0,Coffee Shop,Sushi Restaurant,Japanese Restaurant,Gay Bar,Bubble Tea Shop,Burger Joint,Café,Italian Restaurant,Korean Restaurant,Theater
100,East Toronto,0,Fast Food Restaurant,Boutique,Pizzeria,Café,Smoke Shop,Sandwich Spot,Garden,Garden Center,Deli,Farmers Market


Cluster 1

In [48]:

toronto_merged.loc[toronto_merged['Cluster_Labels'] == 1, toronto_merged.columns[[1] + list(range(5, toronto_merged.shape[1]))]]

,Borough,Cluster_Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
0,North York,1,Park,Food and Beverage Retail,Wings Joint,Electronics Store,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Empanada Restaurant
21,York,1,Park,Bakery,Fishing Store,Wings Joint,Empanada Restaurant,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store
22,Scarborough,1,Korean BBQ Restaurant,Park,Coffee Shop,Escape Room,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store,Empanada Restaurant
35,East York,1,Park,Convenience Store,Rental Car Location,Education,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Electronics Store
52,North York,1,Park,Wings Joint,Empanada Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store,Escape Room
61,Central Toronto,1,Park,Organic Grocery,Bus Line,Wings Joint,Electronics Store,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education
63,York,1,Park,Convenience Store,Furniture and Home Store,Education,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Electronics Store
66,North York,1,Park,Convenience Store,Jewelry Store,Empanada Restaurant,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store,Escape Room
85,Scarborough,1,Park,Playground,Intersection,Wings Joint,Education,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant
91,Downtown Toronto,1,Park,Hiking Trail,Playground,Wings Joint,Electronics Store,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant


Cluster 2

In [49]:
toronto_merged.loc[toronto_merged['Cluster_Labels'] == 2, toronto_merged.columns[[1] + list(range(5, toronto_merged.shape[1]))]]

,Borough,Cluster_Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
51,Scarborough,2,Motel,Harbor or Marina,Electronics Store,Doner Restaurant,Donut Shop,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Wings Joint


Cluster 3

In [50]:
toronto_merged.loc[toronto_merged['Cluster_Labels'] == 3, toronto_merged.columns[[1] + list(range(5, toronto_merged.shape[1]))]]

,Borough,Cluster_Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
50,North York,3,Clothing Store,Wings Joint,Escape Room,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store,Empanada Restaurant,Ethiopian Restaurant
78,Scarborough,3,Clothing Store,Breakfast Spot,Escape Room,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store,Empanada Restaurant,Wings Joint


Cluster 4

In [51]:
toronto_merged.loc[toronto_merged['Cluster_Labels'] == 4, toronto_merged.columns[[1] + list(range(5, toronto_merged.shape[1]))]]

,Borough,Cluster_Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
49,North York,4,Bakery,Wings Joint,Empanada Restaurant,Drugstore,Dumpling Restaurant,Eastern European Restaurant,Education,Electronics Store,Escape Room,Fast Food Restaurant
